In [ ]:
import os
import re
import math
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

import geopandas as gpd
from shapely.geometry import LineString

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import (
    Normalize,
    BoundaryNorm,
    LinearSegmentedColormap
)
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import ScalarFormatter
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple

import seaborn as sns

%matplotlib inline

from sklearn.model_selection import (
    train_test_split,
    learning_curve,
    KFold
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.inspection import (
    permutation_importance,
    PartialDependenceDisplay
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_squared_log_error,
    r2_score,
    accuracy_score,
    classification_report,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score
)

from xgboost import XGBRegressor

import shap
shap.initjs()

import joblib
from tqdm import tqdm

In [ ]:
# Directories and Definitions
BASE_DIR = Path("/Volumes/SI_Drive/Fishnet_Values")
BASE_DIR = Path(BASE_DIR)
FIG_DIR = BASE_DIR / "figures"
TBS_DIR = BASE_DIR / "tables"
MTD_DIR = BASE_DIR / "model_metadata"

TBS_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
MTD_DIR.mkdir(exist_ok=True)

STATIC_DIR = BASE_DIR / "Static Variables"
LUH2_SCENARIO_FOLDERS = {
    "ssp126": BASE_DIR / "ssp126",
    "ssp245": BASE_DIR / "ssp245",
    "ssp370": BASE_DIR / "ssp370",
    "ssp585": BASE_DIR / "ssp585",
}
pr_DIR = BASE_DIR / "bc__pr_shapefiles"
tas_DIR = BASE_DIR / "bc__tas_shapefiles"

YEAR_COLS = np.array([f'Y{year}' for year in np.arange(2014, 2101)])

pr_seasons = ["Annual", "JF", "MAM", "JJAS", "OND"]
tas_seasons = ["Annual", "DJF", "MAM", "JJAS", "ON"]

LUH2_VARIABLE_IDENTIFIERS = [
    "c3ann", "c3nfx", "c3per", "c4ann", "c4per", "pastr", "primf", "primn", "range",
    "secdf", "secdn", "urban", "fertl_c3ann", "fertl_c3nfx", "fertl_c3per",
    "fertl_c4ann", "fertl_c4per", "irrig_c3ann", "irrig_c3nfx", "irrig_c3per",
    "irrig_c4ann", "irrig_c4per", "flood"
]

MODELS = ["BCC-CSM2-MR", "MIROC6", "MPI-ESM1-2-HR", "CanESM5", "ACCESS-CM2",
                 "UKESM1-0-LL", "IITM-ESM", "GFDL-ESM4"]

SCENARIOS = ["ssp126", "ssp245", "ssp370", "ssp585"]

In [ ]:
# Utilities and Data Loading
def list_shapefiles(folder: Path):
    return sorted([p for p in folder.glob("**/*.shp")])

def safe_read_shp(path):
    try:
        return gpd.read_file(path)
    except Exception as e:
        raise RuntimeError(f"Failed to read {path}: {e}")

def ensure_crs_match(gdfs):
    if not gdfs:
        return gdfs
    base_crs = gdfs[0].crs
    for i in range(1, len(gdfs)):
        if gdfs[i].crs != base_crs:
            gdfs[i] = gdfs[i].to_crs(base_crs)
    return gdfs

def year_columns_in_gdf(gdf, year_tokens):
    """Return list of columns in gdf that match any token in year_tokens.
       Matching strategy:
         1) exact match to token (e.g., 'Y2000')
         2) token as substring in column name (case-sensitive)
         3) numeric year substring (fallback) if none found
    """
    cols = list(gdf.columns)
    found = []
    exact_matches = [c for c in cols if c in set(year_tokens)]
    found.extend(exact_matches)
    for token in year_tokens:
        subs = [c for c in cols if (token in c) and (c not in found)]
        found.extend(subs)
    if not found:
        numeric_tokens = [t.lstrip("Y") for t in year_tokens]
        for nt in numeric_tokens:
            subs = [c for c in cols if (nt in c) and (c not in found)]
            found.extend(subs)
    return found

def load_static(static_dir: Path):
    shp_list = list_shapefiles(static_dir)
    if len(shp_list) == 0:
        raise FileNotFoundError(f"No shapefile found in {static_dir}")
    if len(shp_list) > 1:
        print(f"Multiple shapefiles found in {static_dir}; using the first: {shp_list[0].name}")
    static_gdf = safe_read_shp(shp_list[0])
    uid_col = None
    for cand in ("uid", "ID", "id", "OBJECTID"):
        if cand in static_gdf.columns:
            uid_col = cand
            break
    if uid_col is None:
        static_gdf = static_gdf.reset_index(drop=True)
        static_gdf["uid"] = static_gdf.index.astype(str)
        uid_col = "uid"
    static_gdf = static_gdf.set_index(uid_col, drop=False)
    print(f"Loaded static shapefile {shp_list[0].name} — rows: {len(static_gdf)}, uid: {uid_col}")
    return static_gdf, uid_col

def _compile_identifier_patterns(identifiers):
    sorted_idents = sorted(identifiers, key=lambda s: -len(s))
    patterns = []
    for ident in sorted_idents:
        pat = re.compile(r'(?<![A-Za-z0-9])' + re.escape(ident.lower()) + r'(?![A-Za-z0-9])')
        patterns.append((ident, pat))
    return patterns

def load_static(static_dir: Path):
    shp_list = [f for f in static_dir.glob("*.shp") if not f.name.startswith("._")]
    if len(shp_list) == 0:
        raise FileNotFoundError(f"No shapefile found in {static_dir}")
    if len(shp_list) > 1:
        print(f"Multiple shapefiles found in {static_dir}; using the first: {shp_list[0].name}")
    static_gdf = safe_read_shp(shp_list[0])
    uid_col = None
    for cand in ("uid", "ID", "id", "OBJECTID"):
        if cand in static_gdf.columns:
            uid_col = cand
            break
    if uid_col is None:
        static_gdf = static_gdf.reset_index(drop=True)
        static_gdf["uid"] = static_gdf.index.astype(str)
        uid_col = "uid"
    static_gdf = static_gdf.set_index(uid_col, drop=False)
    print(f"Loaded static shapefile {shp_list[0].name} — rows: {len(static_gdf)}, uid: {uid_col}")
    return static_gdf, uid_col

def load_luh2_vars_from_folder(luh2_dir: Path, identifiers, year_tokens=None):
    """Load LUH2 shapefiles from one folder. Returns dict: key -> GeoDataFrame.
       Keys are either the identifier or identifier__<filename-stem> if duplicates found.
    """
    if not luh2_dir.exists():
        print(f"Warning: LUH2 folder {luh2_dir} does not exist; returning empty dict.")
        return {}
    shp_list = list_shapefiles(luh2_dir)
    print(f"Found {len(shp_list)} shapefiles in {luh2_dir}")
    variable_dfs = {}
    ident_patterns = _compile_identifier_patterns(identifiers)
    for p in tqdm(shp_list, desc=f"Loading LUH2 shapefiles in {luh2_dir.name}"):
        fname = p.name.lower()
        matched_ident = None
        for ident, pat in ident_patterns:
            if pat.search(fname):
                matched_ident = ident
                break
        if not matched_ident:
            continue
        var_name = matched_ident
        gdf = safe_read_shp(p)
        if year_tokens is not None:
            cols_to_keep = year_columns_in_gdf(gdf, year_tokens)
        else:
            cols_to_keep = [c for c in gdf.columns if c != gdf.geometry.name]
        if gdf.geometry.name not in cols_to_keep:
            cols_to_keep = cols_to_keep + [gdf.geometry.name]
        subset = gdf[cols_to_keep].copy()
        new_col_names = {}
        for c in subset.columns:
            if c == subset.geometry.name:
                continue
            new_col_names[c] = f"{var_name}__{c}"
        subset = subset.rename(columns=new_col_names)
        key = var_name
        if key in variable_dfs:
            key = f"{var_name}__{p.stem}"
        variable_dfs[key] = subset
    print(f"Loaded {len(variable_dfs)} LUH2 datasets from {luh2_dir.name}.")
    return variable_dfs

def load_all_luh2_scenarios(scenario_folders: dict, identifiers, year_tokens=None):
    """Return:
       - luh2_scenario_dict: mapping scenario -> dict(key->gdf)
       - luh2_all_dict: flattened dict of all scenario datasets with scenario-prefixed keys
    """
    luh2_scenario_dict = {}
    luh2_all_dict = {}
    for scenario, folder in scenario_folders.items():
        scen_dict = load_luh2_vars_from_folder(folder, identifiers, year_tokens=year_tokens)
        scen_prefixed = {}
        for k, gdf in scen_dict.items():
            combined_key = f"luh2__{scenario}__{k}"
            scen_prefixed[combined_key] = gdf
            luh2_all_dict[combined_key] = gdf
        luh2_scenario_dict[scenario] = scen_prefixed
    return luh2_scenario_dict, luh2_all_dict

def parse_bc_filename(fname, seasons_list, models_list, scenarios_list):
    fname_low = fname.lower()
    season = next((s for s in seasons_list if s.lower() in fname_low), None)
    model = next((m for m in models_list if m.lower() in fname_low), None)
    scenario = next((sc for sc in scenarios_list if sc in fname_low), None)
    return model, scenario, season

def load_bc_datasets(dir_path: Path, seasons_list, models_list, scenarios_list, year_tokens=None, var_tag_prefix="pr"):
    shp_list = list_shapefiles(dir_path)
    print(f"Found {len(shp_list)} shapefiles in {dir_path}")
    bc_dfs = {}
    for p in tqdm(shp_list, desc=f"Loading {var_tag_prefix} shapefiles"):
        model, scenario, season = parse_bc_filename(p.name, seasons_list, models_list, scenarios_list)

        if season is None:
            continue
        tag = f"{var_tag_prefix}__{season}__{model or 'modelUnknown'}__{scenario or 'scnUnknown'}"
        gdf = safe_read_shp(p)

        if year_tokens is not None:
            cols_to_keep = year_columns_in_gdf(gdf, year_tokens)
        else:
            cols_to_keep = [c for c in gdf.columns if c != gdf.geometry.name]

        if gdf.geometry.name not in cols_to_keep:
            cols_to_keep = cols_to_keep + [gdf.geometry.name]
        subset = gdf[cols_to_keep].copy()

        new_col_names = {}
        for c in subset.columns:
            if c == subset.geometry.name:
                continue
            new_col_names[c] = f"{tag}__{c}"
        subset = subset.rename(columns=new_col_names)
        bc_dfs[tag] = subset
    print(f"Loaded {len(bc_dfs)} datasets under prefix '{var_tag_prefix}'.")

    return bc_dfs

if __name__ == "__main__":
    print("Loading static data...")
    static_gdf, uid_col = load_static(STATIC_DIR)

    print("Loading LUH2 scenario folders into per-scenario dicts...")
    luh2_scenario_dict, luh2_all_dict = load_all_luh2_scenarios(LUH2_SCENARIO_FOLDERS, LUH2_VARIABLE_IDENTIFIERS, year_tokens=YEAR_COLS)

    print("Loading precipitation (pr) datasets...")
    pr_dict = load_bc_datasets(pr_DIR, pr_seasons, MODELS, SCENARIOS, year_tokens=YEAR_COLS, var_tag_prefix="pr")

    print("Loading temperature (tas) datasets...")
    tas_dict = load_bc_datasets(tas_DIR, tas_seasons, MODELS, SCENARIOS, year_tokens=YEAR_COLS, var_tag_prefix="tas")

print(pd.DataFrame(luh2_all_dict.keys()))
print(pd.DataFrame(pr_dict.keys()))
print(pd.DataFrame(tas_dict.keys()))

In [ ]:
# 2019 DATASET GENERATION (MEAN OF 2014–2018)
PR_OUTPUT_NAMES = {"Annual": "ATP", "JF": "pr_JF", "MAM": "pr_MAM", "JJAS": "pr_JJAS", "OND": "pr_OND"}
TAS_OUTPUT_NAMES = {"Annual": "AMT", "DJF": "tas_DJF", "MAM": "tas_MAM", "JJAS": "tas_JJAS", "ON": "tas_ON"}

YEAR_TOKENS = ["Y2014", "Y2015", "Y2016", "Y2017", "Y2018"]

UID = "OBJECTID"

def _find_year_cols(gdf, tokens=YEAR_TOKENS):
    if "year_columns_in_gdf" in globals():
        cols = []
        for t in tokens:
            cols.extend(year_columns_in_gdf(gdf, [str(t)]))
        return list(dict.fromkeys(cols))

    cols_all = [c for c in gdf.columns if c != gdf.geometry.name]
    matched = []

    for token in tokens:
        token = str(token)

        if token in cols_all:
            matched.append(token)
            continue

        subs = [c for c in cols_all if token in str(c)]
        if subs:
            matched.extend(subs)
            continue

        num = ''.join(ch for ch in token if ch.isdigit())
        if num:
            matched.extend([c for c in cols_all if num in str(c)])

    return list(dict.fromkeys(matched))


def _series_from_gdf(gdf, uid=UID, tokens=YEAR_TOKENS):
    cols = _find_year_cols(gdf, tokens)

    if not cols:
        if uid in gdf.columns:
            idx = gdf[uid].astype(str).values
        else:
            idx = gdf.index.astype(str).values
        return pd.Series(index=idx, data=np.nan)

    vals = gdf[cols].astype(float).mean(axis=1)

    idx = (
        gdf[uid].astype(str).values
        if uid in gdf.columns
        else gdf.index.astype(str).values
    )

    return pd.Series(index=idx, data=vals.values)

def _group_keys_by_scenario(flat_map):
    scen_map = {}
    for k in flat_map.keys():
        kl = k.lower()
        found = None
        for sc in SCENARIOS:
            if sc in kl:
                found = sc
                break
        if found is None:
            found = "unknown"
        scen_map.setdefault(found, []).append(k)
    return scen_map

LUH2_FINAL = {}

if "luh2_scenario_dict" in globals():
    scenario_map = luh2_scenario_dict

elif "luh2_all_dict" in globals():
    flat = luh2_all_dict
    grouped = _group_keys_by_scenario(flat)
    scenario_map = {sc: {k: flat[k] for k in keys} for sc, keys in grouped.items()}

elif "LUH2_SCENARIO_FOLDERS" in globals():
    if "load_luh2_vars_from_folder" in globals():
        scenario_map = {}
        for sc, folder in LUH2_SCENARIO_FOLDERS.items():
            scenario_map[sc] = load_luh2_vars_from_folder(
                folder,
                LUH2_VARIABLE_IDENTIFIERS,
                year_tokens=YEAR_TOKENS
            )
    else:
        raise RuntimeError("LUH2 data not found.")
else:
    raise RuntimeError("LUH2 data structures not found.")

for ident in LUH2_VARIABLE_IDENTIFIERS:
    per_scen = []

    for sc, d in scenario_map.items():
        matched_keys = [k for k in d.keys() if ident.lower() in k.lower()]
        if not matched_keys:
            continue

        series_list = []
        for k in matched_keys:
            s = _series_from_gdf(d[k], uid=UID, tokens=YEAR_TOKENS)
            s.index = s.index.astype(str)
            series_list.append(s)

        if series_list:
            df_tmp = pd.concat(series_list, axis=1)
            per_scen.append(df_tmp.mean(axis=1))

    if per_scen:
        df_allsc = pd.concat(per_scen, axis=1)
        s_final = df_allsc.mean(axis=1)
        s_final.name = ident
        LUH2_FINAL[ident] = s_final
    else:
        LUH2_FINAL[ident] = pd.Series(dtype=float)

def _aggregate_bc(flat_map, seasons, out_name_map):
    res = {}
    scen_map = _group_keys_by_scenario(flat_map)

    for season in seasons:
        per_scen = []

        for sc, keys in scen_map.items():
            keys_sc = [k for k in keys if season.lower() in k.lower()]
            if not keys_sc:
                continue

            s_list = []
            for k in keys_sc:
                s = _series_from_gdf(flat_map[k], uid=UID, tokens=YEAR_TOKENS)
                s.index = s.index.astype(str)
                s_list.append(s)

            if s_list:
                df_models = pd.concat(s_list, axis=1)
                per_scen.append(df_models.mean(axis=1))

        out_name = out_name_map.get(season, season)

        if per_scen:
            df_scen = pd.concat(per_scen, axis=1)
            overall = df_scen.mean(axis=1)
            overall.name = out_name
            res[out_name] = overall
        else:
            res[out_name] = pd.Series(dtype=float)

    return res

PR_FINAL = _aggregate_bc(
    pr_dict,
    pr_seasons,
    {"Annual": "ATP", "JF": "pr_JF", "MAM": "pr_MAM", "JJAS": "pr_JJAS", "OND": "pr_OND"}
)

TAS_FINAL = _aggregate_bc(
    tas_dict,
    tas_seasons,
    {"Annual": "AMT", "DJF": "tas_DJF", "MAM": "tas_MAM", "JJAS": "tas_JJAS", "ON": "tas_ON"}
)

if "static_gdf" not in globals():
    raise RuntimeError("static_gdf not found in workspace.")

if UID in static_gdf.columns:
    static_gdf[UID] = static_gdf[UID].astype(str)
    static_gdf = static_gdf.set_index(UID, drop=False)
else:
    static_gdf.index = static_gdf.index.astype(str)

model_df = pd.DataFrame(
    static_gdf.drop(columns=[static_gdf.geometry.name], errors="ignore")
).copy()
model_df.index = model_df.index.astype(str)

for ident, series in LUH2_FINAL.items():
    model_df[ident] = np.nan if series.empty else series.reindex(model_df.index).values

for name, series in PR_FINAL.items():
    model_df[name] = np.nan if series.empty else series.reindex(model_df.index).values

for name, series in TAS_FINAL.items():
    model_df[name] = np.nan if series.empty else series.reindex(model_df.index).values


MLD_Path = BASE_DIR / "ml_processing"
MLD_Path.mkdir(exist_ok=True)
model_df.to_csv(MLD_Path / "current_scenario_14_19.csv", index=True)

model_df.describe().to_csv(MLD_Path / "current_scenario_14_19_describe.csv", index=True)

print("Final shape:", model_df.shape)

In [ ]:
# FUTURE DATASETS GENERATION
PR_OUTPUT_NAMES = {
    "Annual": "ATP", "JF": "pr_JF", "MAM": "pr_MAM",
    "JJAS": "pr_JJAS", "OND": "pr_OND"
}

TAS_OUTPUT_NAMES = {
    "Annual": "AMT", "DJF": "tas_DJF", "MAM": "tas_MAM",
    "JJAS": "tas_JJAS", "ON": "tas_ON"
}

UID = "OBJECTID"
YEAR_TOKENS = list(range(2015, 2101))
SCENARIOS = SCENARIOS

def _find_year_cols(gdf, tokens):
    cols_all = [c for c in gdf.columns if c != gdf.geometry.name]
    matched = []

    for token in tokens:
        token = str(token)

        if token in cols_all:
            matched.append(token)
            continue

        subs = [c for c in cols_all if token in str(c)]
        if subs:
            matched.extend(subs)

    return list(dict.fromkeys(matched))


def _series_from_gdf(gdf, uid, tokens):
    cols = _find_year_cols(gdf, tokens)

    idx = (
        gdf[uid].astype(str).values
        if uid in gdf.columns
        else gdf.index.astype(str).values
    )

    if not cols:
        return pd.Series(index=idx, data=np.nan)

    vals = gdf[cols].astype(float).mean(axis=1)
    return pd.Series(index=idx, data=vals.values)


def _group_keys_by_scenario(flat_map):
    scen_map = {}
    for k in flat_map.keys():
        kl = k.lower()
        for sc in SCENARIOS:
            if sc in kl:
                scen_map.setdefault(sc, []).append(k)
                break
    return scen_map


def _aggregate_bc_scenario(flat_map, seasons, out_name_map, year_tokens, scenario):
    """MME across models for ONE scenario using multi-year mean"""
    res = {}
    scen_map = _group_keys_by_scenario(flat_map)

    if scenario not in scen_map:
        return {out_name_map[s]: pd.Series(dtype=float) for s in seasons}

    keys_all = scen_map[scenario]

    for season in seasons:
        keys_season = [k for k in keys_all if season.lower() in k.lower()]
        s_list = []

        for k in keys_season:
            s = _series_from_gdf(flat_map[k], uid=UID, tokens=year_tokens)
            s.index = s.index.astype(str)
            s_list.append(s)

        out_name = out_name_map.get(season, season)

        if s_list:
            df_models = pd.concat(s_list, axis=1)
            s_final = df_models.mean(axis=1)
            s_final.name = out_name
            res[out_name] = s_final
        else:
            res[out_name] = pd.Series(dtype=float)

    return res

for YEAR_TOKEN in YEAR_TOKENS:

    YEAR_WINDOW = list(range(YEAR_TOKEN - 5, YEAR_TOKEN))

    print(f"\nProcessing year: {YEAR_TOKEN} | Window: {YEAR_WINDOW[0]}–{YEAR_WINDOW[-1]}")

    if "luh2_scenario_dict" in globals():
        scenario_map = luh2_scenario_dict
    elif "luh2_all_dict" in globals():
        flat = luh2_all_dict
        grouped = _group_keys_by_scenario(flat)
        scenario_map = {sc: {k: flat[k] for k in keys} for sc, keys in grouped.items()}
    else:
        raise RuntimeError("LUH2 data not found.")

    for SC in SCENARIOS:

        print(f"  → Scenario: {SC}")

        if SC not in scenario_map:
            continue

        d_sc = scenario_map[SC]

        LUH2_FINAL = {}

        for ident in LUH2_VARIABLE_IDENTIFIERS:
            matched_keys = [k for k in d_sc.keys() if ident.lower() in k.lower()]
            s_list = []

            for k in matched_keys:
                s = _series_from_gdf(d_sc[k], uid=UID, tokens=YEAR_WINDOW)
                s.index = s.index.astype(str)
                s_list.append(s)

            if s_list:
                df_tmp = pd.concat(s_list, axis=1)
                s_final = df_tmp.mean(axis=1)
                s_final.name = ident
                LUH2_FINAL[ident] = s_final
            else:
                LUH2_FINAL[ident] = pd.Series(dtype=float)

        PR_FINAL = _aggregate_bc_scenario(
            pr_dict, pr_seasons, PR_OUTPUT_NAMES, YEAR_WINDOW, SC
        )

        TAS_FINAL = _aggregate_bc_scenario(
            tas_dict, tas_seasons, TAS_OUTPUT_NAMES, YEAR_WINDOW, SC
        )

        if UID in static_gdf.columns:
            static_gdf[UID] = static_gdf[UID].astype(str)
            static_gdf = static_gdf.set_index(UID, drop=False)
        else:
            static_gdf.index = static_gdf.index.astype(str)

        model_df = pd.DataFrame(
            static_gdf.drop(columns=[static_gdf.geometry.name], errors="ignore")
        ).copy()
        model_df.index = model_df.index.astype(str)

        for ident, series in LUH2_FINAL.items():
            model_df[ident] = series.reindex(model_df.index).values

        for name, series in PR_FINAL.items():
            model_df[name] = series.reindex(model_df.index).values

        for name, series in TAS_FINAL.items():
            model_df[name] = series.reindex(model_df.index).values

        model_df.to_csv(MLD_Path / f"futures_{YEAR_TOKEN}_{SC}.csv", index=True)

        print(f"    Saved → {MLD_Path} | Shape: {model_df.shape}")

print("\nAll YEAR × SCENARIO files generated successfully.")


In [ ]:
# PIML Modelling
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11
})

plt.rcParams.update({
    "font.family": "Times New Roman",
    "mathtext.fontset": "custom",
    "mathtext.rm": "Times New Roman",
    "mathtext.it": "Times New Roman:italic",
    "mathtext.bf": "Times New Roman:bold"
})

dataset = pd.read_csv(BASE_DIR / "ml_processing/current_scenario_14_19.csv")
dataset = dataset.dropna().copy()
dataset.drop(columns=["SOC", "SOC__10_9", "Soil_N"], inplace=True, errors="ignore")

df = dataset[dataset["Stock"] > 0].copy()

X = df.drop(columns=["Stock", "OBJECTID", "OBJECTID.1"], errors="ignore")
y = np.log(df["Stock"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

monotonic_constraints = [
-1, -1, +1, 0, 0, 0, +1, 0, -1,
0, -1, -1, -1, -1, +1, +1, +1, +1,
+1, +1, +1, +1, +1, +1, +1, -1, +1,
0, +1, +1, +1, 0, 0, 0, 0, 0, +1,
+1, +1, +1, +1, +1, -1, -1, -1, -1, -1
]

assert len(monotonic_constraints) == X.shape[1]
monotone_constraints = "(" + ",".join(map(str, monotonic_constraints)) + ")"

xgb_constrained = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=600,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=10,
    gamma=0.5,
    subsample=0.8,
    colsample_bytree=0.8,
    monotone_constraints=monotone_constraints,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_unconstrained = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=600,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=10,
    gamma=0.5,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_constrained.fit(X_train, y_train)
xgb_unconstrained.fit(X_train, y_train)

def report(name, yt, yp):
    print(f"{name:25s} | RMSE={np.sqrt(mean_squared_error(yt,yp)):.3f} | R²={r2_score(yt,yp):.3f}")

print("\n=== LOG-SCALE PERFORMANCE ===")
report("Train (Constrained)", y_train, xgb_constrained.predict(X_train))
report("Test  (Constrained)", y_test,  xgb_constrained.predict(X_test))
report("Train (Unconstrained)", y_train, xgb_unconstrained.predict(X_train))
report("Test  (Unconstrained)", y_test,  xgb_unconstrained.predict(X_test))

performance_rows = [
    {
        "Model": "XGB_Constrained",
        "Dataset": "Train",
        "RMSE_logSOC": np.sqrt(mean_squared_error(
            y_train, xgb_constrained.predict(X_train)
        )),
        "R2_logSOC": r2_score(
            y_train, xgb_constrained.predict(X_train)
        )
    },
    {
        "Model": "XGB_Constrained",
        "Dataset": "Test",
        "RMSE_logSOC": np.sqrt(mean_squared_error(
            y_test, xgb_constrained.predict(X_test)
        )),
        "R2_logSOC": r2_score(
            y_test, xgb_constrained.predict(X_test)
        )
    },
    {
        "Model": "XGB_Unconstrained",
        "Dataset": "Train",
        "RMSE_logSOC": np.sqrt(mean_squared_error(
            y_train, xgb_unconstrained.predict(X_train)
        )),
        "R2_logSOC": r2_score(
            y_train, xgb_unconstrained.predict(X_train)
        )
    },
    {
        "Model": "XGB_Unconstrained",
        "Dataset": "Test",
        "RMSE_logSOC": np.sqrt(mean_squared_error(
            y_test, xgb_unconstrained.predict(X_test)
        )),
        "R2_logSOC": r2_score(
            y_test, xgb_unconstrained.predict(X_test)
        )
    }
]

performance_df = pd.DataFrame(performance_rows)

performance_df.to_csv(
    TBS_DIR / "log_scale_model_performance.csv",
    index=False
)

residuals = y_train - xgb_constrained.predict(X_train)
smearing_factor = np.mean(np.exp(residuals))

print("\n=== SMEARING CORRECTION ===")
print(f"Smearing factor: {smearing_factor:.4f}")

def soc_corrected(model, X):
    return np.exp(model.predict(X)) * smearing_factor

smearing_df = pd.DataFrame([{
    "Smearing_Factor": smearing_factor
}])

smearing_df.to_csv(
    TBS_DIR / "smearing_correction_factor.csv",
    index=False
)

observed_2019 = df["Stock"].sum()
pred_2019_raw = np.exp(xgb_constrained.predict(X)).sum()
pred_2019_corr = soc_corrected(xgb_constrained, X).sum()
anchor_offset = observed_2019 - pred_2019_corr

print("\n=== NATIONAL SOC (2019) ===")
print(f"Observed:          {observed_2019:,.2f}")
print(f"Predicted (raw):   {pred_2019_raw:,.2f}")
print(f"Predicted (corr):  {pred_2019_corr:,.2f}")
print(f"Anchoring offset:  {anchor_offset:,.2f}")
print(f"Bias raw (%):      {(pred_2019_raw/observed_2019-1)*100:.2f}")
print(f"Bias corr (%):     {(pred_2019_corr/observed_2019-1)*100:.2f}")

national_soc_2019 = pd.DataFrame([{
    "Observed_SOC_2019": observed_2019,
    "Predicted_SOC_Raw_2019": pred_2019_raw,
    "Predicted_SOC_Corrected_2019": pred_2019_corr,
    "Anchoring_Offset": anchor_offset,
    "Bias_Raw_percent": (pred_2019_raw / observed_2019 - 1) * 100,
    "Bias_Corrected_percent": (pred_2019_corr / observed_2019 - 1) * 100
}])

national_soc_2019.to_csv(
    TBS_DIR / "national_soc_2019_accounting_correction.csv",
    index=False
)

def soc_final(model, X):
    return soc_corrected(model, X) + anchor_offset / len(X)

N_ENSEMBLE = 30
ensemble_models = []

for seed in range(N_ENSEMBLE):
    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=600,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=10,
        gamma=0.5,
        subsample=0.8,
        colsample_bytree=0.8,
        monotone_constraints=monotone_constraints,
        tree_method="hist",
        random_state=seed,
        n_jobs=-1
    )

    idx = np.random.choice(len(X_train), len(X_train), replace=True)
    model.fit(X_train.iloc[idx], y_train.iloc[idx])
    ensemble_models.append(model)

ensemble_preds = np.column_stack([
    m.predict(X_test) for m in ensemble_models
])

ensemble_df = pd.DataFrame(
    ensemble_preds,
    columns=[f"Ensemble_{i+1:02d}" for i in range(N_ENSEMBLE)]
)

ensemble_df["Ensemble_Mean"] = ensemble_preds.mean(axis=1)
ensemble_df["Ensemble_SD"]   = ensemble_preds.std(axis=1)
ensemble_df["CV_percent"]    = (
    ensemble_df["Ensemble_SD"] /
    np.abs(ensemble_df["Ensemble_Mean"])
) * 100
ensemble_df["Observed_logSOC"] = y_test.values

ensemble_df.to_csv(
    TBS_DIR / "ensemble_member_predictions.csv",
    index=False
)

summary_table = ensemble_df[
    ["Ensemble_Mean", "Ensemble_SD", "CV_percent"]
].agg(["mean", "std", "min", "max"])

summary_table.to_csv(
    TBS_DIR / "ensemble_uncertainty_summary.csv"
)

order = np.argsort(ensemble_df["Ensemble_Mean"].values)
ensemble_sorted = ensemble_df.iloc[order].reset_index(drop=True)

plt.figure(figsize=(3, 3))
x = np.arange(len(ensemble_sorted))

for i in range(N_ENSEMBLE):
    plt.plot(
        x,
        ensemble_sorted[f"Ensemble_{i+1:02d}"],
        color="grey",
        alpha=0.15,
        lw=0.5,
        label="Ensemble Members" if i == 0 else None
    )

plt.plot(
    x,
    ensemble_sorted["Ensemble_Mean"],
    color="black",
    lw=1.5,
    label="PIML Model"
)

plt.fill_between(
    x,
    ensemble_sorted["Ensemble_Mean"] - ensemble_sorted["Ensemble_SD"],
    ensemble_sorted["Ensemble_Mean"] + ensemble_sorted["Ensemble_SD"],
    color="blue",
    alpha=0.75,
    label="±1 SD"
)

plt.scatter(
    x,
    ensemble_sorted["Observed_logSOC"],
    s=6,
    alpha=0.15,
    color="red",
    label="Observed Values"
)

plt.xlabel("Test Data Samples")
plt.ylabel(r"$\it{log\ (\text{SOC}_{stock})}$")
plt.legend(frameon=True, edgecolor = "red", facecolor="white", fontsize=10, fancybox=False)
plt.tight_layout()

plt.savefig(
    FIG_DIR / "ensemble_uncertainty_all_members.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

quantile_models = {}
for q in [0.25, 0.75]:
    qm = XGBRegressor(
        objective="reg:quantileerror",
        quantile_alpha=q,
        n_estimators=600,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )
    qm.fit(X_train, y_train)
    quantile_models[q] = qm

q25 = quantile_models[0.25].predict(X)
q75 = quantile_models[0.75].predict(X)

quantile_uncertainty = pd.DataFrame({
    "Q25_logSOC": q25,
    "Q75_logSOC": q75,
    "IQR_logSOC": q75 - q25
})

quantile_uncertainty.to_csv(
    TBS_DIR / "quantile_uncertainty.csv"
)

quantile_uncertainty.describe().to_csv(
    TBS_DIR / "quantile_uncertainty_summary.csv"
)

MODEL_DIR = BASE_DIR / "saved_models"
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(
    {
        "model": xgb_constrained,
        "smearing_factor": smearing_factor,
        "anchor_offset": anchor_offset,
        "feature_names": X.columns.tolist()
    },
    MODEL_DIR / "soc_corrected_pixgboost_model.joblib"
)

print("\nFinal corrected SOC model saved.")

def ensemble_predict(models, X):
    """
    Generate ensemble mean and standard deviation
    in log space from a list of trained models.
    """
    preds = np.column_stack([
        m.predict(X) for m in models
    ])
    return preds.mean(axis=1), preds.std(axis=1)

INPUT_DIR = BASE_DIR / "ml_processing"

for scnn in SCENARIOS:
    CSV_FILES = sorted(INPUT_DIR.glob(f"futures*{scnn}.csv"))
    base_df = None

    for csv in CSV_FILES:
        dfi = pd.read_csv(csv).dropna()

        if base_df is None:
            base_df = dfi[["OBJECTID"]].copy()

        X_new = dfi[X.columns]

        ens_mean, ens_std = ensemble_predict(ensemble_models, X_new)

        q25 = quantile_models[0.25].predict(X_new)
        q75 = quantile_models[0.75].predict(X_new)

        year = int(re.search(r"(19|20|21)\d{2}", csv.name).group()) + 5

        base_df[f"Y{year}_mean"] = np.exp(ens_mean) * smearing_factor + anchor_offset/len(X_new)
        base_df[f"Y{year}_std"] = (np.exp(ens_mean) * ens_std * smearing_factor)
        base_df[f"Y{year}_q25"]  = np.exp(q25) * smearing_factor + anchor_offset/len(X_new)
        base_df[f"Y{year}_q75"]  = np.exp(q75) * smearing_factor + anchor_offset/len(X_new)

    OUT = BASE_DIR / f"predictions/{scnn}_soc_with_uncertainty_corrected.csv"
    OUT.parent.mkdir(exist_ok=True)
    base_df.to_csv(OUT, index=False)

print("\nAll corrected future projections generated successfully.")

def hexbin_kde(y_true, y_pred, out_path=None, title_suffix="", dpi=300):
    lims = [12.5, 16.5]
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    plt.figure(figsize=(3, 3))

    hb = plt.hexbin(y_true, y_pred, gridsize=45, cmap="RdGy_r", mincnt=1)
    sns.kdeplot(x=y_true, y=y_pred, levels=6, cmap="Blues", linewidths=1.5)

    plt.plot(lims, lims, "--k", lw=2)
    plt.xlim(lims); plt.ylim(lims)

    plt.xlabel(r"Observed $\it{log\ (\text{SOC}_{stock})}$")
    plt.ylabel(r"Predicted $\it{log\ (\text{SOC}_{stock})}$")

    plt.text(
        0.95, 0.05,
        f"RMSE={rmse:.2f}\n$R^2$={r2:.2f}",
        transform=plt.gca().transAxes,
        ha="right", va="bottom",
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="red")
    )

    plt.colorbar(hb, label="Data Density")
    plt.tight_layout()

    if out_path is not None:
        plt.savefig(out_path, dpi=dpi, bbox_inches="tight")

    plt.show()

hexbin_kde(
    y_train,
    xgb_constrained.predict(X_train),
    out_path=FIG_DIR / "hexbin_kde_train.png", dpi=600,
    title_suffix="(Training)"
)

hexbin_kde(
    y_test,
    xgb_constrained.predict(X_test),
    out_path=FIG_DIR / "hexbin_kde_test.png", dpi=600,
    title_suffix="(Testing)"
)

def scatter_kde(
    y_true,
    y_pred,
    point_color,
    out_path=None,
    title_suffix="",
    dpi=300
):
    lims = [12.5, 16.5]
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    plt.figure(figsize=(3, 3))

    plt.scatter(
        y_true,
        y_pred,
        s=10,
        alpha=0.45,
        color=point_color,
        edgecolor="none"
    )

    sns.kdeplot(
        x=y_true,
        y=y_pred,
        levels=6,
        cmap="Greys_r",
        linewidths=1.2
    )

    plt.plot(lims, lims, "--k", lw=1.5)
    plt.xlim(lims)
    plt.ylim(lims)

    plt.xlabel(r"Observed $\it{log\ (\text{SOC}_{stock})}$")
    plt.ylabel(r"Predicted $\it{log\ (\text{SOC}_{stock})}$")

    plt.text(
        0.95,
        0.05,
        f"RMSE={rmse:.2f}\n$R^2$={r2:.2f}",
        transform=plt.gca().transAxes,
        ha="right",
        va="bottom",
        fontsize=10,
        bbox=dict(
            facecolor="white",
            alpha=0.85,
            edgecolor="red"
        )
    )

    plt.tight_layout()

    if out_path is not None:
        plt.savefig(out_path, dpi=dpi, bbox_inches="tight")

    plt.show()

scatter_kde(
    y_train,
    xgb_constrained.predict(X_train),
    point_color="tab:blue",
    out_path=FIG_DIR / "scatter_kde_train.png",
    dpi=600,
    title_suffix="(Training)"
)

scatter_kde(
    y_test,
    xgb_constrained.predict(X_test),
    point_color="tab:red",
    out_path=FIG_DIR / "scatter_kde_test.png",
    dpi=600,
    title_suffix="(Testing)"
)

explainer = shap.TreeExplainer(xgb_constrained)
shap_values = explainer.shap_values(X_train)

shap_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "MeanAbsSHAP": np.abs(shap_values).mean(axis=0)
}).sort_values("MeanAbsSHAP", ascending=False)

shap_importance.to_csv(
    TBS_DIR / "shap_feature_importance.csv",
    index=False
)

features_tas = [
    "AMT", "tas_DJF", "tas_MAM", "tas_JJAS", "tas_ON"
]

features_pr = [
    "ATP", "pr_JF", "pr_MAM", "pr_JJAS", "pr_OND"
]

plt.rcParams["font.family"] = "Times New Roman"

X_shap = X_train.copy()
X_shap.columns = X_shap.columns.str.replace("_", "_")

shap_df = pd.DataFrame(
    shap_values,
    columns=X_shap.columns
)

features_tas = [f for f in features_tas if f in X_shap.columns]
features_pr  = [f for f in features_pr  if f in X_shap.columns]

def beeswarm_subplot(ax, shap_df, X_shap, features, xlabel):

    cmap = cm.get_cmap("jet")
    n_feat = len(features)

    for i in range(n_feat):
        ax.axhline(i, color="lightgrey", linestyle="dotted",
                   linewidth=1, zorder=0)
    ax.axvline(0, color="black", linestyle="--",
               linewidth=1, zorder=0)

    for i, feature in enumerate(reversed(features)):
        shap_vals = shap_df[feature].values
        feature_vals = X_shap[feature].values
        y_vals = np.random.normal(i, 0.15, size=len(shap_vals))

        vmin, vmax = np.nanmin(feature_vals), np.nanmax(feature_vals)
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        colors = cmap(norm(feature_vals))

        ax.scatter(
            shap_vals, y_vals,
            c=colors,
            s=3,
            alpha=1,
            edgecolor="none",
            zorder=3
        )

    ax.set_yticks(range(n_feat))
    ax.set_yticklabels(list(reversed(features)), fontsize=12)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.tick_params(labelsize=11)

    return norm, cmap

fig, axes = plt.subplots(
    nrows=2, ncols=1,
    figsize=(4, 5.25),
    sharex=False
)

norm_tas, cmap = beeswarm_subplot(
    axes[0], shap_df, X_shap, features_tas,
    xlabel="SHAP value"
)

norm_pr, _ = beeswarm_subplot(
    axes[1], shap_df, X_shap, features_pr,
    xlabel="SHAP value"
)

sm = cm.ScalarMappable(cmap=cmap, norm=norm_pr)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=axes[1],
    fraction=0.046,
    pad=0.05
)

cbar.set_ticks([norm_pr.vmin, norm_pr.vmax])
cbar.set_ticklabels(["Low", "High"])
cbar.set_label("Feature value", fontsize=12)
cbar.ax.tick_params(labelsize=11)

sm = cm.ScalarMappable(cmap=cmap, norm=norm_pr)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=axes[0],
    fraction=0.046,
    pad=0.05
)

cbar.set_ticks([norm_pr.vmin, norm_pr.vmax])
cbar.set_ticklabels(["Low", "High"])
cbar.set_label("Feature value", fontsize=12)
cbar.ax.tick_params(labelsize=11)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "shap_beeswarm_climate_panels.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

cv = KFold(n_splits=5, shuffle=True, random_state=42)
train_sizes = np.linspace(0.1, 1.0, 6)

def plot_lc(model, label, color):
    ts, tr, va = learning_curve(
        model, X, y,
        train_sizes=train_sizes,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )
    plt.plot(ts, -tr.mean(axis=1), 'o-', color=color, label=f"Train ({label})")
    plt.plot(ts, -va.mean(axis=1), 's--', color=color, label=f"Val ({label})")

plt.figure(figsize=(4, 3))
plot_lc(xgb_constrained, "PIML", "black")
plot_lc(xgb_unconstrained, "XGBoost", "tab:blue")
plt.xlabel("Training Sample Size")
plt.ylabel(r"RMSE of $\it{log\ (\text{SOC}_{stock})}$")
plt.legend(frameon=True, edgecolor = "red", facecolor="white", fontsize=9, fancybox=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "learning_curves.png", dpi=600)
plt.show()

plt.figure(figsize=(3, 3))

y_pred_test = xgb_constrained.predict(X_test)
residuals  = y_test - y_pred_test

plt.scatter(
    y_pred_test,
    residuals,
    marker="s",
    s=12,
    alpha=0.45,
    color="red"
)

sns.kdeplot(
    x=y_pred_test,
    y=residuals,
    levels=5,
    cmap="Greys_r",
    linewidths=1.5,
    alpha=1
)

plt.axhline(0, color="black", lw=1.2)

plt.xlabel(r"Predicted $\it{log\ (\text{SOC}_{stock})}$")
plt.ylabel("Residuals")
plt.gca().set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.savefig(FIG_DIR / "residuals_vs_fitted.png", dpi=600)

plt.show()

plt.figure(figsize=(3.5, 3.5))

y_obs = df["Stock"].values

y_corr = soc_final(xgb_constrained, X)

y_uncorr = np.exp(xgb_constrained.predict(X))

r2_corr   = r2_score(y_obs, y_corr)
r2_uncorr = r2_score(y_obs, y_uncorr)

plt.scatter(
    y_obs,
    y_corr,
    zorder=2,
    s=10,
    alpha=0.25,
    color="green",
    label=f"$R^2_{{Corrected}}$ = {r2_corr:.2f}"
)

plt.scatter(
    y_obs,
    y_uncorr,
    zorder=1,
    s=10,
    alpha=0.25,
    color="red",
    label=f"$R^2_{{log-Bias}}$ = {r2_uncorr:.2f}"
)

sns.kdeplot(
    x=y_obs,
    y=y_corr,
    levels=5,
    cmap="Greys_r",
    linewidths=1.5,
    alpha=0.8
)

lims = [
    min(y_obs.min(), y_corr.min(), y_uncorr.min()),
    max(y_obs.max(), y_corr.max(), y_uncorr.max())
]

plt.plot(lims, lims, "--k", lw=1.5)

formatter = ScalarFormatter(useMathText=True)
formatter.set_powerlimits((6, 6))
ax = plt.gca()
ax.set_xlim(0, 1e7)
ax.set_ylim(0, 1e7)
ax.xaxis.set_major_formatter(formatter)
ax.yaxis.set_major_formatter(formatter)
ax.ticklabel_format(axis="both", style="sci", scilimits=(6, 6))

plt.xlabel(r"Observed $\it{\text{SOC}_{stock}}$ (Mg C)")
plt.ylabel(r"Predicted $\it{\text{SOC}_{stock}}$ (Mg C)")

ax.set_aspect("equal", adjustable="box")
plt.legend(frameon=True, edgecolor="red", fontsize=10, handletextpad=0.2)
plt.tight_layout()

plt.savefig(FIG_DIR / "obs_vs_corrected_vs_uncorrected.png", dpi=600)
plt.show()

In [ ]:
# PIML PROJECTION and PLOTTING
BASE_DIR = Path(BASE_DIR)
PRED_DIR = BASE_DIR / "predictions"

plt.rcParams["font.family"] = "Times New Roman"

def extract_years(cols, suffix):
    return sorted([
        int(re.findall(r"\d{4}", c)[0])
        for c in cols if c.endswith(suffix)
    ])

if "OBJECTID" in static_gdf.index.names:
    static_gdf = static_gdf.reset_index(drop=True)

static_gdf["OBJECTID"] = static_gdf["OBJECTID"].astype(str)
baseline_2019_sum = static_gdf["Stock"].sum()

plt.figure(figsize=(4.5, 3.5))
ax = plt.gca()

lines = []
labels = ["SSP126", "SSP245", "SSP370", "SSP585"]

for scn, color in zip(labels, ["green", "blue", "orange", "red"]):

    df = pd.read_csv(PRED_DIR / f"{scn}_soc_with_uncertainty_corrected.csv")
    df["OBJECTID"] = df["OBJECTID"].astype(str)
    df = df.merge(static_gdf[["OBJECTID"]], on="OBJECTID", how="inner")

    years = extract_years(df.columns, "_mean")
    mean_vals = [df[f"Y{y}_mean"].sum() for y in years]

    years_full = [2019] + years
    mean_full  = [baseline_2019_sum] + mean_vals

    line, = ax.plot(
        years_full,
        mean_full,
        lw=2.5,
        marker="|",
        markersize=3,
        color=color
    )
    lines.append(line)

vertical_lines = [2019, 2045, 2070]
vline_colors = ['teal', 'dodgerblue', 'blue']
vlabels = ['2019', '2045', '2070']

for vline, color, lbl in zip(vertical_lines, vline_colors, vlabels):
    ax.axvline(vline, color=color, lw=1.2, ls='--')
    ax.annotate(
        lbl,
        xy=(vline, 0.03),
        xycoords=('data', 'axes fraction'),
        ha='center',
        va='bottom',
        rotation=90,
        fontsize=10,
        color='white',
        bbox=dict(facecolor=color, edgecolor='none', boxstyle='round,pad=0.15')
    )

label_positions = {
    "SSP126": (2080, 15.00e9),
    "SSP245": (2091, 14.65e9),
    "SSP370": (2091, 14.325e9),
    "SSP585": (2060, 14.25e9),
}

for line, name in zip(lines, labels):
    x_lbl, y_lbl = label_positions[name]
    ax.annotate(
        name,
        xy=(x_lbl, y_lbl),
        color='white',
        fontsize=10,
        fontweight='demibold',
        ha='center',
        va='center',
        bbox=dict(facecolor=line.get_color(), edgecolor='none',
                  boxstyle='round,pad=0.25', alpha=0.8)
    )

regimes = [
    {"start": 2020, "end": 2045, "color": "teal",       "label": "Early"},
    {"start": 2045, "end": 2070, "color": "dodgerblue", "label": "Mid"},
    {"start": 2070, "end": 2100, "color": "blue",       "label": "Late"}
]

for reg in regimes:
    ax.annotate(
        reg["label"],
        xy=((reg["start"] + reg["end"]) / 2, 1.03),
        xycoords=("data", "axes fraction"),
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="demibold",
        color="white",
        bbox=dict(facecolor=reg["color"], edgecolor="none",
                  boxstyle="round,pad=0.25"),
        clip_on=False
    )

formatter = ScalarFormatter(useMathText=True)
formatter.set_powerlimits((9, 9))
ax.tick_params(labelsize=14)
ax.yaxis.set_major_formatter(formatter)
ax.set_yticks([14e9, 14.5e9, 15e9])
ax.set_ylabel(r"$\it{\text{SOC}_{stock}}$ (Mg C)", fontsize=14)
ax.grid(alpha=0.4, linestyle="--")

plt.tight_layout()
plt.savefig(FIG_DIR / "soc_stock_all.png", dpi=600)
plt.show()

fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(4.5, 6.5), sharex=True)

for ax, scn, color in zip(
    axes,
    ["SSP126", "SSP245", "SSP370", "SSP585"],
    ["green", "blue", "orange", "red"]
):

    df = pd.read_csv(PRED_DIR / f"{scn}_soc_with_uncertainty_corrected.csv")
    df["OBJECTID"] = df["OBJECTID"].astype(str)
    df = df.merge(static_gdf[["OBJECTID"]], on="OBJECTID", how="inner")

    years = extract_years(df.columns, "_mean")

    mean_vals = [df[f"Y{y}_mean"].sum() for y in years]
    q25_vals  = [df[f"Y{y}_q25"].sum()  for y in years]
    q75_vals  = [df[f"Y{y}_q75"].sum()  for y in years]
    std_vals  = [df[f"Y{y}_std"].sum()  for y in years]

    years_full = [2019] + years
    mean_full  = [baseline_2019_sum] + mean_vals
    q25_full   = [baseline_2019_sum] + q25_vals
    q75_full   = [baseline_2019_sum] + q75_vals
    std_full   = [0.0] + std_vals

    ax.fill_between(
        years_full,
        [m - s for m, s in zip(mean_full, std_full)],
        [m + s for m, s in zip(mean_full, std_full)],
        color="grey",
        alpha=0.25,
        zorder=1
    )

    ax.fill_between(
        years_full,
        q25_full,
        q75_full,
        color=color,
        alpha=0.25,
        zorder=2
    )

    ax.plot(years_full, mean_full, lw=2.5, color=color, zorder=3)

    ax.annotate(
        scn,
        xy=(years_full[-20], 16e9),
        xytext=(6, 0),
        textcoords='offset points',
        color='white',
        fontsize=10,
        fontweight='demibold',
        ha='left',
        va='center',
        bbox=dict(facecolor=color, edgecolor='none',
                  boxstyle='round,pad=0.25')
    )

    for vline, vcolor in zip([2019, 2045, 2070], ['teal', 'dodgerblue', 'blue']):
        ax.axvline(vline, color=vcolor, lw=1.2, ls='--')

    for reg in regimes:
        ax.annotate(
            reg["label"] if scn == "SSP126" else None,
            xy=((reg["start"] + reg["end"]) / 2, 1.08),
            xycoords=("data", "axes fraction"),
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="demibold",
            color="white",
            bbox=dict(facecolor=reg["color"], edgecolor="none",
                      boxstyle="round,pad=0.25"),
            clip_on=False
        )

    ax.yaxis.set_major_formatter(formatter)
    ax.tick_params(labelsize=14)
    ax.set_yticks([13.5e9, 15e9, 17e9])
    ax.grid(alpha=0.4, linestyle="--")

fig.supylabel(r"$\it{\text{SOC}_{stock}}$ (Mg C)", x=0.06, fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / "soc_stock_scenarios.png", dpi=600)
plt.show()